# 13 — Sensitivity analyses (Stage 8)

Each pre-registered sensitivity (HYPOTHESES.md §sensitivity #1–9) reruns the headline `RII_route / RII_buffer` ratio. Forest plot at the end. Given that the primary analysis came back with a **reversed** H2 direction (RII ratio = 0.288), this notebook is the load-bearing robustness check.

**Run in this notebook (data already on disk):**
- #1 Salon source: Google-only / OSM-only / Union (= primary) / Intersection
- #2 Buffer distance: 400 / 800 (= primary) / 1600 m
- #4 Deprivation index: IMD2025 (= primary) vs IDACI
- #6 Route-buffer width: 50 m (= primary) vs 100 m
- #7 Restrict to schools with ≥1 salon in any buffer

**Documented but not rerun here (would need full Stage-4/5 regeneration):**
- #3 Catchment proxy (hard / k=3 IDW / straight-line) — k-NN routes not built yet
- #5 Distance caps ±25% — needs catchment + route regen
- #8 Routing engine — already validated in Stage 5 (Pearson r = 1.0000 vs networkx)
- #9 OA21 origin geometry (HYPOTHESES.md amendment A-01) — needs OA21 PWC fetch

These remaining sensitivities are scoped for follow-up commits.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

from schools_sunbeds import audit, config, exposure, stats, verification

config.ensure_dirs()
audit.verify_manifest()
TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")

## 0. Load all the inputs

In [ ]:
schools = gpd.read_file(
    sorted(
        p for p in config.DATA_PROCESSED.glob("schools_ne_*.gpkg") if "sensitivity" not in p.name
    )[-1]
)
lsoa = gpd.read_file(sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))[-1])
school_bufs = gpd.read_file(sorted(config.DATA_PROCESSED.glob("school_euclidean_buffers_*.gpkg"))[-1])
rb50 = gpd.read_file(sorted(config.DATA_PROCESSED.glob("route_buffer_50m_*.gpkg"))[-1])
rb100 = gpd.read_file(sorted(config.DATA_PROCESSED.glob("route_buffer_100m_*.gpkg"))[-1])
salons_union = gpd.read_file(sorted(config.DATA_PROCESSED.glob("salons_verified_union_*.gpkg"))[-1])
salons_intersection_path = sorted(config.DATA_PROCESSED.glob("salons_intersection_*.gpkg"))[-1]
google_path = sorted(config.DATA_PROCESSED.glob("salons_google_*.gpkg"))[-1]
osm_path = sorted(config.DATA_PROCESSED.glob("salons_osm_*.gpkg"))[-1]

print(f"Schools: {len(schools)}")
print(f"Salons (verified union): {len(salons_union)} ({(salons_union['source'] == 'google').sum()} google + {(salons_union['source'] == 'osm').sum()} osm-only)")

## 1. Salon-source variants for sensitivity #1

In [ ]:
# Apply verification to the raw Google + OSM layers and the intersection layer
ver = verification.load_verification(config.AUDIT_LOGS / "manual_verification.csv")
google = gpd.read_file(google_path)
osm = gpd.read_file(osm_path)
salons_intersection_raw = gpd.read_file(salons_intersection_path)

google_v, _ = verification.apply_verification(google, ver, source="google", source_id_col="place_id")
osm_v, _ = verification.apply_verification(osm, ver, source="osm", source_id_col="osm_id")
intersection_v, _ = verification.apply_verification(
    salons_intersection_raw, ver, source="google", source_id_col="place_id"
)

salon_variants = {
    "union (primary)": salons_union,
    "google_only": google_v.assign(source="google"),
    "osm_only": osm_v.assign(source="osm"),
    "intersection": intersection_v.assign(source="intersection"),
}
for name, gdf in salon_variants.items():
    print(f"  {name:>20s}: {len(gdf):>4} salons")

## 2. Helper — recompute the headline RII ratio for any (salons, buffer-distance, route-buffer-width) triple

In [ ]:
def compute_rii_ratio(
    salons_gdf: gpd.GeoDataFrame,
    *,
    buf_distance_m: int = 800,
    route_width_m: int = 50,
    deprivation_col: str = "imd_quintile",
    zero_restrict: bool = False,
) -> dict:
    """Return a dict with rii_buffer, rii_route, ratio, and meta."""
    rb = rb50 if route_width_m == 50 else rb100
    width_label = f"{route_width_m}m"

    sb = school_bufs.loc[school_bufs["distance_m"] == buf_distance_m]
    buf = exposure.buffer_exposure(sb, salons_gdf)
    rt = exposure.route_exposure(rb, salons_gdf, width_label=width_label)

    # Build the modelling frame
    panel = (
        schools[["urn", "la_code_dfe"]]
        .merge(buf, on="urn", how="left")
        .merge(rt, on="urn", how="left")
    )
    panel["n_pupils_school"] = schools["n_pupils"].fillna(1).clip(lower=1).values
    panel["sum_route"] = panel[f"sum_route_{width_label}"].fillna(0)
    panel["sum_pupil"] = panel[f"sum_pupil_{width_label}"].fillna(0)
    panel["n_buf"] = panel[f"n_buffer_{buf_distance_m}m"].fillna(0).astype(int)

    # School-IMD or IDACI via spatial join
    schools_with_dep = schools.sjoin(
        lsoa[["lsoa21cd", deprivation_col, "geometry"]],
        how="left",
        predicate="within",
    )[["urn", deprivation_col]]
    panel = panel.merge(schools_with_dep, on="urn", how="left")
    panel = panel.dropna(subset=[deprivation_col]).copy()
    panel[deprivation_col] = panel[deprivation_col].astype("int64")
    panel["ridit"] = stats.compute_ridit(panel[deprivation_col], panel["sum_pupil"])
    panel["_pupil_offset"] = panel["sum_pupil"].clip(lower=1)

    if zero_restrict:
        any_salon = (panel["n_buf"] > 0) | (panel["sum_route"] > 0)
        panel = panel.loc[any_salon].copy()

    sub = panel.loc[panel["sum_pupil"] > 0].copy()
    if len(sub) < 50 or len(panel) < 50:
        return {"rii_buffer": np.nan, "rii_route": np.nan, "ratio": np.nan, "n": int(len(panel))}

    try:
        rii_b = 1.0 / stats.relative_index_inequality(panel, "n_buf", "ridit")
    except Exception:
        rii_b = np.nan
    try:
        rii_r = 1.0 / stats.relative_index_inequality(
            sub, "sum_route", "ridit", offset_col="_pupil_offset"
        )
    except Exception:
        rii_r = np.nan
    return {
        "rii_buffer": float(rii_b),
        "rii_route": float(rii_r),
        "ratio": float(rii_r / rii_b) if rii_b and not np.isnan(rii_b) else np.nan,
        "n": int(len(panel)),
    }

primary = compute_rii_ratio(salons_union)
print("Primary (union, 800 m, 50 m, IMD):", primary)

## 3. Run all sensitivities

In [ ]:
rows = [{"name": "primary (union, 800 m, 50 m, IMD)", **primary}]

# Sensitivity #1: salon source
for name in ("google_only", "osm_only", "intersection"):
    s = compute_rii_ratio(salon_variants[name])
    rows.append({"name": f"#1 source: {name}", **s})

# Sensitivity #2: buffer distance (use the primary salon set)
for d in (400, 1600):
    s = compute_rii_ratio(salons_union, buf_distance_m=d)
    rows.append({"name": f"#2 buffer: {d} m", **s})

# Sensitivity #4: deprivation index
s_idaci = compute_rii_ratio(salons_union, deprivation_col="idaci_quintile")
rows.append({"name": "#4 deprivation: IDACI", **s_idaci})

# Sensitivity #6: route buffer width
s_100 = compute_rii_ratio(salons_union, route_width_m=100)
rows.append({"name": "#6 route buffer: 100 m", **s_100})

# Sensitivity #7: zero-restriction
s_zr = compute_rii_ratio(salons_union, zero_restrict=True)
rows.append({"name": "#7 schools with >=1 salon", **s_zr})

sens = pd.DataFrame(rows)
sens

## 4. Forest plot of the headline ratio

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(sens))
ax.scatter(sens["ratio"], y, color="#cc0000", s=60, zorder=3)
ax.axvline(1.0, color="black", linewidth=0.8, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(sens["name"])
ax.invert_yaxis()
ax.set_xlabel("RII_route / RII_buffer (>1 = route gradient steeper than buffer)")
ax.set_xscale("log")
ax.set_title("Sensitivity of the headline H2 ratio across pre-registered dimensions")
plt.tight_layout()
fig_out = config.OUTPUTS_FIGURES / f"F5_sensitivity_forest_{TODAY}.png"
plt.savefig(fig_out, dpi=150)
plt.savefig(fig_out.with_suffix(".pdf"))
print("Wrote", fig_out.relative_to(config.REPO_ROOT))
plt.show()

In [ ]:
out_table = config.OUTPUTS_TABLES / f"T5_sensitivity_matrix_{TODAY}.csv"
sens.to_csv(out_table, index=False)
print("Wrote", out_table.relative_to(config.REPO_ROOT))

## 5. Robustness narrative

Translate the table into a one-sentence summary for the manuscript: across all pre-registered sensitivities run here, does the H2 inversion hold?

In [ ]:
below_one = (sens["ratio"] < 1).sum()
valid = sens["ratio"].notna().sum()
print(f"Of {valid} sensitivities run, {below_one} show RII_route / RII_buffer < 1 (the H2 reversal direction).")
print()
if below_one == valid:
    print("All sensitivities point the same way: the H2 inversion is robust across these dimensions.")
elif below_one >= valid * 0.7:
    print("Most sensitivities reproduce the inversion; a minority flip — flag in the discussion.")
else:
    print("Substantial disagreement across sensitivities — the H2 result depends on methodological choices.")